# ContractRisk Colombia — M1: Fine-tuning con LoRA

Este notebook entrena un clasificador que ayuda a analistas y auditores a priorizar descripciones contractuales de SECOP II que no permiten comprender claramente qué se está contratando.

**Tarea:** descripción contractual → `REQUIERE_REVISION` (0) o `SUFICIENTE` (1). La predicción no implica fraude, corrupción ni irregularidad.

## 1. Entorno reproducible

En Colab seleccione **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4**. Las versiones se fijan para evitar cambios incompatibles de API.

In [4]:
!pip -q install transformers==4.48.1 datasets==3.2.0 peft==0.14.0 accelerate==1.2.1 scikit-learn==1.5.2 sentencepiece==0.2.0

import os, random, numpy as np, pandas as pd, torch
SEED=42
os.environ["PYTHONHASHSEED"]=str(SEED)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print("PyTorch:",torch.__version__,"| GPU:",torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

PyTorch: 2.11.0+cu128 | GPU: Tesla T4


## 2. Datos anotados

Suba `contractrisk_train.csv` y `contractrisk_validation.csv` al entorno de Colab. El dataset final contiene 699 textos únicos: 559 para entrenamiento y 140 para validación, con split estratificado, semilla 42 y cero fuga por ID o texto.

In [5]:
from google.colab import files
from pathlib import Path

ARCHIVOS_DATOS = ["contractrisk_train.csv", "contractrisk_validation.csv"]
if any(not Path(nombre).exists() for nombre in ARCHIVOS_DATOS):
    print("Seleccione contractrisk_train.csv y contractrisk_validation.csv")
    files.upload()

train_df = pd.read_csv("contractrisk_train.csv", encoding="utf-8-sig")
val_df = pd.read_csv("contractrisk_validation.csv", encoding="utf-8-sig")

columnas_requeridas = {"id_contrato", "text", "label", "label_text"}
assert columnas_requeridas.issubset(train_df.columns)
assert columnas_requeridas.issubset(val_df.columns)
assert not train_df[list(columnas_requeridas)].isna().any().any()
assert not val_df[list(columnas_requeridas)].isna().any().any()
assert not train_df.id_contrato.duplicated().any()
assert not val_df.id_contrato.duplicated().any()
assert not train_df.text.duplicated().any()
assert not val_df.text.duplicated().any()
assert set(train_df.id_contrato).isdisjoint(val_df.id_contrato)
assert set(train_df.text).isdisjoint(val_df.text)
assert set(train_df.label) == {0, 1} and set(val_df.label) == {0, 1}

label_a_texto = {0: "REQUIERE_REVISION", 1: "SUFICIENTE"}
assert train_df.apply(lambda fila: label_a_texto[fila.label] == fila.label_text, axis=1).all()
assert val_df.apply(lambda fila: label_a_texto[fila.label] == fila.label_text, axis=1).all()
assert len(train_df) == 559 and len(val_df) == 140

print("Train:", train_df.shape, "| Validation:", val_df.shape)
display(train_df.label_text.value_counts(), val_df.label_text.value_counts())

(559, 4) (140, 4)


,count
label_text,
SUFICIENTE,435
REQUIERE_REVISION,124


,count
label_text,
SUFICIENTE,109
REQUIERE_REVISION,31


## 3. Selección del encoder

Se comparan tres encoders con preentrenamiento de lenguaje enmascarado, apropiados para aprender representaciones bidireccionales y resolver clasificación de secuencias:

| Modelo | Alcance | Ventaja | Consideración |
|---|---|---|---|
| `BSC-LT/RoBERTalex` | Español jurídico | Dominio cercano a contratos y lenguaje administrativo | Modelo base viable en T4 con LoRA |
| `dccuchile/bert-base-spanish-wwm-cased` | Español general | BETO con whole-word masking | Tokenizador WordPiece y arquitectura BERT |
| `FacebookAI/xlm-roberta-base` | Multilingüe | Cobertura de numerosos idiomas | Capacidad multilingüe innecesaria para un corpus solo en español |

Como evidencia auxiliar se compara cuántos subtokens necesitan términos contractuales representativos. Una menor fragmentación puede producir secuencias más eficientes, aunque no demuestra por sí sola cuál modelo tendrá mejor desempeño; la elección también considera idioma, dominio, arquitectura y costo computacional.

In [6]:
from transformers import AutoTokenizer

candidatos = [
    "BSC-LT/RoBERTalex",
    "dccuchile/bert-base-spanish-wwm-cased",
    "FacebookAI/xlm-roberta-base",
]

terminos = [
    "interventoría",
    "licitación pública",
    "gestión catastral multipropósito",
    "arrendamiento de inmuebles",
    "mantenimiento preventivo y correctivo",
    "apoyo a la gestión",
]

rows = []

for model_id in candidatos:
    tok = AutoTokenizer.from_pretrained(model_id)

    for termino in terminos:
        pieces = tok.tokenize(termino)
        rows.append({
            "modelo": model_id,
            "termino": termino,
            "subtokens": len(pieces),
            "segmentacion": " | ".join(pieces),
        })

tokenizacion_df = pd.DataFrame(rows)
display(tokenizacion_df)
display(
    tokenizacion_df
    .groupby("modelo")["subtokens"]
    .agg(["mean", "max"])
    .sort_values("mean")
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/638 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

,modelo,termino,subtokens,segmentacion
0,BSC-LT/RoBERTalex,interventoría,2,Ġintervent | orÃŃa
1,BSC-LT/RoBERTalex,licitación pública,2,ĠlicitaciÃ³n | ĠpÃºblica
2,BSC-LT/RoBERTalex,gestión catastral multipropósito,5,ĠgestiÃ³n | Ġcatastral | Ġmulti | prop | Ã³sito
3,BSC-LT/RoBERTalex,arrendamiento de inmuebles,3,Ġarrendamiento | Ġde | Ġinmuebles
4,BSC-LT/RoBERTalex,mantenimiento preventivo y correctivo,4,Ġmantenimiento | Ġpreventivo | Ġy | Ġcorrectivo
5,BSC-LT/RoBERTalex,apoyo a la gestión,4,Ġapoyo | Ġa | Ġla | ĠgestiÃ³n
6,dccuchile/bert-base-spanish-wwm-cased,interventoría,2,interven | ##toría
7,dccuchile/bert-base-spanish-wwm-cased,licitación pública,2,licitación | pública
8,dccuchile/bert-base-spanish-wwm-cased,gestión catastral multipropósito,7,gestión | catas | ##tral | multi | ##pro | ##p...
9,dccuchile/bert-base-spanish-wwm-cased,arrendamiento de inmuebles,3,arrendamiento | de | inmuebles


,mean,max
modelo,,
BSC-LT/RoBERTalex,3.333333,5
dccuchile/bert-base-spanish-wwm-cased,4.000000,7
FacebookAI/xlm-roberta-base,4.666667,8


### Modelo seleccionado

Se selecciona `BSC-LT/RoBERTalex`, un encoder RoBERTa especializado en español jurídico. En la comparación exploratoria obtuvo la menor fragmentación promedio de los términos contractuales evaluados (3.33 subtokens, frente a 4.00 de BETO y 4.67 de XLM-R). Su dominio es más cercano al lenguaje contractual que el de los otros candidatos y su tamaño base es compatible con una GPU T4 mediante LoRA.

La comparación de tokenización es una evidencia complementaria, no una evaluación de clasificación. Los resultados predictivos se determinan posteriormente sobre el conjunto de validación congelado.

## 4. Baselines reproducibles

Los dos puntos de partida se calculan sobre exactamente las mismas 140 filas de validación usadas por el Transformer:

- **Baseline 0 — clase mayoritaria:** predice siempre la clase más frecuente de train (`SUFICIENTE`). Sirve como mínimo trivial.
- **Baseline 1 — TF-IDF + Logistic Regression:** representa unigramas y bigramas, descarta términos presentes en menos de dos documentos, limita el vocabulario a 10.000 características y utiliza pesos balanceados para compensar el desbalance sin modificar las etiquetas.

La métrica principal es **Macro F1** porque calcula el F1 de cada clase y les asigna el mismo peso. También se reportan accuracy, Macro Precision, Macro Recall, reportes por clase y matrices de confusión.

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
)


def evaluar_predicciones(nombre, y_real, y_predicho):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_real, y_predicho, average="macro", zero_division=0
    )
    return {
        "modelo": nombre,
        "accuracy": accuracy_score(y_real, y_predicho),
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1,
    }


y_train = train_df["label"].to_numpy()
y_val = val_df["label"].to_numpy()

# Baseline 0: siempre predice la clase más frecuente observada únicamente en train.
clase_mayoritaria = int(train_df["label"].mode()[0])
pred_mayoria = np.full(shape=len(val_df), fill_value=clase_mayoritaria, dtype=int)
resultado_mayoria = evaluar_predicciones("Clase mayoritaria", y_val, pred_mayoria)
cm_mayoria = confusion_matrix(y_val, pred_mayoria, labels=[0, 1])

# Baseline 1: el vectorizador se ajusta solo con train para evitar fuga.
tfidf_logreg = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_features=10_000,
    )),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=1_000,
        random_state=SEED,
    )),
])

tfidf_logreg.fit(train_df["text"], y_train)
pred_tfidf = tfidf_logreg.predict(val_df["text"])
resultado_tfidf = evaluar_predicciones(
    "TF-IDF + Logistic Regression", y_val, pred_tfidf
)
cm_tfidf = confusion_matrix(y_val, pred_tfidf, labels=[0, 1])

resultados_baselines = pd.DataFrame([resultado_mayoria, resultado_tfidf])
display(resultados_baselines)
print("Matriz — clase mayoritaria:\n", cm_mayoria)
print("Matriz — TF-IDF + Logistic Regression:\n", cm_tfidf)
print(classification_report(
    y_val,
    pred_tfidf,
    labels=[0, 1],
    target_names=["REQUIERE_REVISION", "SUFICIENTE"],
    zero_division=0,
))

# Evidencia fila por fila y matrices reutilizables en la exportación final.
predicciones_tfidf = val_df[["id_contrato", "text", "label", "label_text"]].copy()
predicciones_tfidf["pred_label"] = pred_tfidf
predicciones_tfidf["pred_text"] = [label_a_texto[x] for x in pred_tfidf]
predicciones_tfidf["correcto"] = predicciones_tfidf["label"] == predicciones_tfidf["pred_label"]

                         modelo  accuracy  macro_precision  macro_recall  macro_f1
0             Clase mayoritaria  0.778571         0.389286      0.500000  0.437751
1  TF-IDF + Logistic Regression  0.857143         0.790476      0.815922  0.801756
Matriz — clase mayoritaria:
 [[  0  31]
 [  0 109]]
Matriz — TF-IDF + Logistic Regression:
 [[23  8]
 [12 97]]
                   precision    recall  f1-score   support

REQUIERE_REVISION       0.66      0.74      0.70        31
       SUFICIENTE       0.92      0.89      0.91       109

         accuracy                           0.86       140
        macro avg       0.79      0.82      0.80       140
     weighted avg       0.86      0.86      0.86       140


### Lectura de los baselines

| Modelo | Accuracy | Macro Precision | Macro Recall | Macro F1 |
|---|---:|---:|---:|---:|
| Clase mayoritaria | 0.779 | 0.389 | 0.500 | 0.438 |
| TF-IDF + Logistic Regression | 0.857 | 0.790 | 0.816 | **0.802** |

El baseline mayoritario obtiene una accuracy aparentemente alta debido al desbalance, pero no detecta ningún caso `REQUIERE_REVISION`. TF-IDF detecta 23 de los 31 casos de revisión, con 8 falsos negativos y 12 falsos positivos. Por eso, 0.802 de Macro F1 constituye el baseline razonable contra el cual se compara LoRA.

## 5. Tokenización y Dataset de Hugging Face

In [8]:
from datasets import Dataset
MODEL_ID = "BSC-LT/RoBERTalex"
MAX_LENGTH=256
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,use_fast=True)

train_ds=Dataset.from_pandas(train_df[["text","label"]],preserve_index=False)
val_ds=Dataset.from_pandas(val_df[["text","label"]],preserve_index=False)
def tokenize_batch(batch):
    return tokenizer(batch["text"],truncation=True,max_length=MAX_LENGTH)
train_tok=train_ds.map(tokenize_batch,batched=True,remove_columns=["text"])
val_tok=val_ds.map(tokenize_batch,batched=True,remove_columns=["text"])

lengths=[len(tokenizer(t,truncation=False)["input_ids"]) for t in pd.concat([train_df.text,val_df.text])]
print(pd.Series(lengths).describe(percentiles=[.90,.95,.99]))
print("Porcentaje truncado:",np.mean(np.array(lengths)>MAX_LENGTH))

Map:   0%|          | 0/559 [00:00<?, ? examples/s]

Map:   0%|          | 0/140 [00:00<?, ? examples/s]

count    699.000000
mean      45.406295
std       17.280370
min        5.000000
50%       47.000000
90%       67.000000
95%       72.000000
99%       78.020000
max       93.000000
dtype: float64
Porcentaje truncado: 0.0


## 6. Modelo y configuración LoRA

- `r=8`: capacidad suficiente para 699 ejemplos sin añadir demasiados parámetros.
- `alpha=16`: escala `alpha/r=2`, una adaptación moderada.
- `dropout=0.10`: regularización ante un dataset pequeño.
- `target_modules=["query","value"]`: adapta las proyecciones de consulta y valor de cada bloque de atención.
- `modules_to_save=["classifier"]`: entrena y conserva la cabeza binaria nueva.
- `bias="none"`: minimiza parámetros entrenables.

In [9]:
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig,TaskType,get_peft_model

id2label={0:"REQUIERE_REVISION",1:"SUFICIENTE"}; label2id={v:k for k,v in id2label.items()}
base_model=AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,num_labels=2,id2label=id2label,label2id=label2id
)
lora_config=LoraConfig(
    task_type=TaskType.SEQ_CLS,r=8,lora_alpha=16,lora_dropout=0.10,
    target_modules=["query","value"],bias="none",modules_to_save=["classifier"]
)
model=get_peft_model(base_model,lora_config)
model.print_trainable_parameters()

pytorch_model.bin:   0%|          | 0.00/504M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at BSC-LT/RoBERTalex and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/504M [00:00<?, ?B/s]

trainable params: 887,042 || all params: 126,866,692 || trainable%: 0.6992


## 7. Entrenamiento con Trainer

Se evalúa al final de cada época, se conserva el checkpoint con mejor Macro F1 y se utiliza precisión mixta FP16 en GPU.

In [10]:
from sklearn.metrics import accuracy_score,precision_recall_fscore_support,confusion_matrix,classification_report
from transformers import DataCollatorWithPadding,TrainingArguments,Trainer,EarlyStoppingCallback

def compute_metrics(eval_pred):
    logits,labels=eval_pred; preds=np.argmax(logits,axis=-1)
    p,r,f1,_=precision_recall_fscore_support(labels,preds,average="macro",zero_division=0)
    return {"accuracy":accuracy_score(labels,preds),"macro_precision":p,"macro_recall":r,"macro_f1":f1}

args=TrainingArguments(
    output_dir="contractrisk_robertalex_lora",eval_strategy="epoch",save_strategy="epoch",
    learning_rate=2e-4,per_device_train_batch_size=16,per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,num_train_epochs=8,weight_decay=0.01,warmup_ratio=0.10,
    fp16=torch.cuda.is_available(),logging_steps=10,load_best_model_at_end=True,
    metric_for_best_model="macro_f1",greater_is_better=True,save_total_limit=2,
    seed=SEED,data_seed=SEED,report_to="none"
)
trainer=Trainer(model=model,args=args,train_dataset=train_tok,eval_dataset=val_tok,
    processing_class=tokenizer,data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
train_result=trainer.train()
train_result.metrics

Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1
1,0.531500,0.490987,0.821429,0.746148,0.689109,0.709085
2,0.487500,0.400324,0.850000,0.853175,0.684374,0.721986
3,0.431400,0.365977,0.857143,0.886735,0.688961,0.730354
4,0.358800,0.341538,0.857143,0.886735,0.688961,0.730354
5,0.348700,0.328777,0.850000,0.833669,0.695916,0.731531
6,0.349400,0.318576,0.857143,0.861333,0.700503,0.739874
7,0.290400,0.316704,0.857143,0.842659,0.712045,0.748563


{'train_runtime': 43.2602,
 'train_samples_per_second': 103.374,
 'train_steps_per_second': 3.144,
 'total_flos': 158802251770296.0,
 'train_loss': 0.3882974824484657,
 'epoch': 7.571428571428571}

## 8. Evaluación final, errores y ejemplos cualitativos

In [11]:
pred_output = trainer.predict(val_tok)
pred = np.argmax(pred_output.predictions, axis=-1)
y_true = val_df["label"].to_numpy()
metrics = compute_metrics((pred_output.predictions, y_true))
print(metrics)

cm = confusion_matrix(y_true, pred, labels=[0, 1])
print(cm)
print(classification_report(
    y_true,
    pred,
    labels=[0, 1],
    target_names=[id2label[0], id2label[1]],
    zero_division=0,
))

# La comparación usa objetos calculados en las celdas de baseline; no hay métricas escritas a mano.
resultado_transformer = {
    "modelo": "RoBERTalex + LoRA",
    "accuracy": metrics["accuracy"],
    "macro_precision": metrics["macro_precision"],
    "macro_recall": metrics["macro_recall"],
    "macro_f1": metrics["macro_f1"],
}
resultados_finales = pd.DataFrame([
    resultado_mayoria,
    resultado_tfidf,
    resultado_transformer,
])
display(resultados_finales)

predicciones_transformer = val_df[["id_contrato", "text", "label", "label_text"]].copy()
predicciones_transformer["pred_label"] = pred
predicciones_transformer["pred_text"] = [id2label[x] for x in pred]
predicciones_transformer["correcto"] = (
    predicciones_transformer["label"] == predicciones_transformer["pred_label"]
)
display(predicciones_transformer.loc[~predicciones_transformer.correcto].head(10))

# Tres casos trazables: un acierto por clase y un falso negativo relevante.
ids_ejemplos = [
    "CO1.PCCNTR.2445956",  # Acierto SUFICIENTE
    "CO1.PCCNTR.7396140",  # Acierto REQUIERE_REVISION
    "CO1.PCCNTR.4323708",  # Fallo: revisión predicha como suficiente
]
ejemplos_cualitativos = predicciones_transformer[
    predicciones_transformer["id_contrato"].isin(ids_ejemplos)
].set_index("id_contrato").loc[ids_ejemplos].reset_index()
display(ejemplos_cualitativos[["id_contrato", "text", "label_text", "pred_text", "correcto"]])

{'accuracy': 0.8571428571428571, 'macro_precision': np.float64(0.8426590148254424), 'macro_recall': np.float64(0.712044983722995), 'macro_f1': np.float64(0.7485632183908046)}
[[ 14  17]
 [  3 106]]
                   precision    recall  f1-score   support

REQUIERE_REVISION       0.82      0.45      0.58        31
       SUFICIENTE       0.86      0.97      0.91       109

         accuracy                           0.86       140
        macro avg       0.84      0.71      0.75       140
     weighted avg       0.85      0.86      0.84       140



## 9. Guardado del adapter y resultados

In [12]:
ADAPTER_DIR = "contractrisk_robertalex_lora_adapter"
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

resultados_finales.to_csv(
    "contractrisk_resultados_finales.csv", index=False, encoding="utf-8-sig"
)
predicciones_transformer.to_csv(
    "contractrisk_predicciones_transformer.csv", index=False, encoding="utf-8-sig"
)
predicciones_tfidf.to_csv(
    "contractrisk_predicciones_tfidf_logreg.csv", index=False, encoding="utf-8-sig"
)

pd.DataFrame(
    cm,
    index=["real_revision", "real_suficiente"],
    columns=["pred_revision", "pred_suficiente"],
).to_csv("contractrisk_matriz_confusion_transformer.csv", encoding="utf-8-sig")

pd.DataFrame(
    cm_tfidf,
    index=["real_revision", "real_suficiente"],
    columns=["pred_revision", "pred_suficiente"],
).to_csv("contractrisk_matriz_confusion_tfidf_logreg.csv", encoding="utf-8-sig")

!zip -qr contractrisk_robertalex_lora_adapter.zip "$ADAPTER_DIR"
!zip -q contractrisk_resultados_m1.zip contractrisk_resultados_finales.csv contractrisk_predicciones_transformer.csv contractrisk_predicciones_tfidf_logreg.csv contractrisk_matriz_confusion_transformer.csv contractrisk_matriz_confusion_tfidf_logreg.csv

files.download("contractrisk_robertalex_lora_adapter.zip")
files.download("contractrisk_resultados_m1.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 10. Criterio de interpretación

La comparación final se interpreta después de obtener las métricas sobre validation. Se informa tanto si LoRA supera los baselines como si no lo hace, prestando especial atención a los falsos negativos de `REQUIERE_REVISION`, porque representan descripciones insuficientes que el sistema dejaría pasar sin revisión.

## 11. Interpretación final y análisis cualitativo

RoBERTalex + LoRA alcanzó **accuracy 0.857** y **Macro F1 0.749**. Superó claramente la clase mayoritaria (Macro F1 0.438), pero no superó TF-IDF + regresión logística (0.802): el delta frente al baseline razonable fue **-0.053**. Por tanto, el experimento demuestra un fine-tuning funcional y eficiente, no una mejora del mejor baseline.

La matriz de confusión fue `[[14, 17], [3, 106]]`. El modelo reconoció bien `SUFICIENTE` (recall 0.972), pero solo detectó 14 de 31 descripciones `REQUIERE_REVISION` (recall 0.452). Los 17 falsos negativos son el riesgo operativo principal porque enviarían descripciones insuficientes sin revisión.

### Ejemplos cualitativos trazables

| ID | Entrada resumida | Real | Predicción | Lectura |
|---|---|---|---|---|
| `CO1.PCCNTR.2445956` | “Fortalecer el programa de vigilancia de calidad del agua mediante análisis microbiológicos y fisicoquímicos...” | SUFICIENTE | SUFICIENTE | Acierto: identifica actividad, objeto y finalidad. |
| `CO1.PCCNTR.7396140` | “Apoyo a la gestión ... como auxiliar en enfermería dentro de la estrategia EBS...” | REQUIERE_REVISION | REQUIERE_REVISION | Acierto: profesión y contexto sin una función contractual concreta. |
| `CO1.PCCNTR.4323708` | “Servicios profesionales especializados para fortalecer actividades propias de la Superintendencia...” | REQUIERE_REVISION | SUFICIENTE | Fallo: el lenguaje institucional parece informativo, pero no concreta la actividad. |

Los tres IDs se seleccionan explícitamente en la celda de evaluación, por lo que la tabla es reproducible y verificable contra `contractrisk_predicciones_transformer.csv`.

### Limitaciones observadas

- Solo 155 de 699 ejemplos pertenecen a `REQUIERE_REVISION`.
- El muestreo fue enriquecido por tipos y posibles casos difíciles, por lo que no reproduce la distribución natural completa de SECOP II.
- Las etiquetas dependen de una guía humana y existen casos frontera entre profesión/área y actividad concreta.
- El modelo base fue preentrenado con español jurídico general, no específicamente con descripciones SECOP colombianas.
- La comparación de tokenización se realizó con seis expresiones y sirve únicamente como evidencia auxiliar.
- Antes de uso real debe priorizarse aumentar el recall de `REQUIERE_REVISION`, por ejemplo con pérdida ponderada, sobremuestreo solo en train, ajuste de umbral y más ejemplos minoritarios.